In [2]:
!pip install playwright nest_asyncio
!playwright install chromium
!playwright install-deps chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 19.3 MB/s eta 0:00:00
177 MiB [] 0% 359.8s177 MiB [] 0% 250.1s177 MiB [] 0% 261.0s177 MiB [] 0% 801.4s177 MiB [] 0% 671.8s177 MiB [] 0% 929.5s177 MiB [] 0% 817.8s177 MiB [] 0% 741.3s177 MiB [] 0% 684.0s177 MiB [] 0% 638.1s177 MiB [] 0% 630.8s177 MiB [] 0% 598.1s177 MiB [] 0% 571.8s177 MiB [] 0% 543.4s177 MiB [] 0% 524.0s177 MiB [] 0% 505.6s177 MiB [] 0% 488.1s177 MiB [] 0% 472.0s177 MiB [] 0% 456.4s177 MiB [] 0% 442.4s177 MiB [] 0% 413.2s177 MiB [] 0% 388.5s177 MiB [] 0% 365.9s177 MiB [] 0% 347.5s177 MiB [] 0% 331.3s177 MiB [] 0% 316.8s177 MiB [] 0% 303.7s177 MiB [] 0% 295.7s177 MiB [] 0% 289.0s177 MiB [] 0% 272.2s177 MiB [] 0% 258.3s177 MiB [] 0% 245.6s177 MiB [] 0% 234.1s177 MiB [] 0% 223.8s177 MiB [] 0% 211.4s177 MiB [] 0% 200.4s177 MiB [] 0% 190.7s177 MiB [] 0% 180.8s177 MiB [] 0% 171.2s177 MiB [] 0% 164.5s177 MiB [] 0% 156.8s177 MiB [] 0% 149.8s177 MiB [] 0% 142.1s177 MiB [] 0% 135.5s177 MiB [] 0% 129.3s177 MiB [] 0% 122.8s1

#Testes da API

In [ ]:
"""
descobrir_api_colab.py

Versão adaptada para Google Colab:
- headless=True obrigatório (Colab não tem display gráfico)
- usa nest_asyncio para permitir rodar dentro do event loop do notebook
- pode ser rodado com `await main()` numa célula do Colab

Antes de rodar, instale (em uma célula separada, uma vez por sessão):
    !pip install playwright nest_asyncio
    !playwright install chromium
    !playwright install-deps chromium
"""

import nest_asyncio
nest_asyncio.apply()

from playwright.async_api import async_playwright
import json
import asyncio

URL_PARTIDA = "https://lnb.com.br/partidas/nbb-2022-2023-flamengo-x-123-minas-01032023-2000/"

PALAVRAS_CHAVE = ["stat", "player", "jogador", "quarto", "quarter", "min", "boxscore", "api", "json"]


def parece_relevante(url: str) -> bool:
    url_lower = url.lower()
    return any(p in url_lower for p in PALAVRAS_CHAVE)


async def main():
    respostas_capturadas = []

    async with async_playwright() as p:
        # headless=True é obrigatório no Colab (não há display gráfico)
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
            )
        )

        async def on_response(response):
            url = response.url
            content_type = response.headers.get("content-type", "")
            if "json" in content_type or parece_relevante(url):
                try:
                    status = response.status
                    print(f"\n[{status}] {url}")
                    if "json" in content_type:
                        try:
                            body = await response.json()
                            preview = json.dumps(body, ensure_ascii=False)[:500]
                            print(f"  -> JSON preview: {preview}")
                            respostas_capturadas.append({"url": url, "body": body})
                        except Exception:
                            pass
                except Exception as e:
                    print(f"  (erro lendo resposta: {e})")

        page.on("response", lambda r: asyncio.create_task(on_response(r)))

        print(f"Navegando até: {URL_PARTIDA}")
        # networkidle costuma travar em páginas com widgets "ao vivo" que
        # nunca param de fazer requisições em segundo plano. Usamos
        # domcontentloaded (mais confiável) + espera fixa em seguida.
        await page.goto(URL_PARTIDA, wait_until="domcontentloaded", timeout=60000)
        await page.wait_for_timeout(5000)  # dá tempo do JS inicial rodar

        for texto_botao in ["1ºQ", "2ºQ", "3ºQ", "4ºQ", "TODOS"]:
            try:
                await page.get_by_text(texto_botao, exact=True).first.click(timeout=3000)
                await page.wait_for_timeout(1500)
            except Exception:
                print(f"  (não consegui clicar no botão '{texto_botao}' - talvez o seletor precise ajuste)")

        await page.wait_for_timeout(3000)

        # Tira um screenshot pra você conferir visualmente o que carregou
        # (útil no Colab, já que você não vê o navegador abrindo)
        await page.screenshot(path="screenshot_partida.png", full_page=True)
        print("\nScreenshot salvo em screenshot_partida.png (confira se a página carregou certo)")

        await browser.close()

    print("\n\n=== RESUMO DE CHAMADAS JSON CAPTURADAS ===")
    for r in respostas_capturadas:
        print(r["url"])

    with open("respostas_capturadas.json", "w", encoding="utf-8") as f:
        json.dump(respostas_capturadas, f, ensure_ascii=False, indent=2)
    print("\nSalvo em respostas_capturadas.json")

    return respostas_capturadas

In [ ]:
respostas = await main()

Navegando até: https://lnb.com.br/partidas/nbb-2022-2023-flamengo-x-123-minas-01032023-2000/

[200] https://lnb.com.br/partidas/nbb-2022-2023-flamengo-x-123-minas-01032023-2000/

[200] https://lnb.com.br/wp-content/uploads/2016/10/minas_2.png

[200] https://static.cloudflareinsights.com/beacon.min.js/v4513226cdae34746b4dedf0b4dfa099e1781791509496

[200] https://lnb.com.br/wp-content/uploads/2026/01/fluminense-1.png

[200] https://fonts.gstatic.com/s/materialicons/v33/2fcrYFNaTjcS6g4U3t-Y5ZjZjT5FdEJ140U2DJYC3mY.woff2

[200] https://fonts.gstatic.com/s/lato/v14/tI4j516nok_GrVf4dhunkg.woff2

[200] https://fonts.gstatic.com/s/lato/v14/1YwB1sO8YE1Lyjf12WNiUA.woff2

[200] https://fonts.gstatic.com/s/lato/v14/EsvMC5un3kjyUhB9ZEPPwg.woff2

[200] https://fonts.gstatic.com/s/lato/v14/H2DMvhDLycM56KNuAtbJYA.woff2

[200] https://lnb.com.br/wp-content/uploads/2016/10/logo-minas-1-150x150.png

[200] https://lnb.com.br/wp-content/themes/lnb-2016/images/icon-statistics-white.png

[200] https://lnb.com

In [ ]:
"""
pipeline_completo_colab.py

Pipeline completo para extrair minutos por atleta por quarto de TODOS os
jogos do KTO Minas na temporada 2022/2023 do NBB.

Estratégia:
1. Abre a "tabela de jogos" filtrada para o Minas / temporada 2022-2023
   e coleta os links /partidas/... de cada jogo.
2. Para cada partida, abre a página com Playwright e intercepta
   especificamente a chamada à API:
       https://lnb.com.br/ws/tempo_real_bybr/json/{ID}_tempo_real.json
3. Parseia o JSON de cada jogo e monta uma tabela única com:
   jogo_id, data, equipe, jogador, quarto, tempo_mmss, tempo_segundos

Antes de rodar no Colab (uma vez por sessão):
    !pip install playwright nest_asyncio pandas
    !playwright install chromium
    !playwright install-deps chromium

Uso no Colab:
    df = await main()
"""

import nest_asyncio
nest_asyncio.apply()

from playwright.async_api import async_playwright
import asyncio
import json
import pandas as pd

SEASON_URL_TABELA = (
    "https://lnb.com.br/nbb/tabela-de-jogos/"
    "?season%5B%5D=71&team%5B%5D=NTQ%3D&wherePlaying=-1&played=-1"
)

QUARTO_LABELS = ["Total", "1ºQ", "2ºQ", "3ºQ", "4ºQ"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)


def tempo_para_segundos(tempo_str: str) -> int:
    if not tempo_str:
        return 0
    m, s = tempo_str.split(":")
    return int(m) * 60 + int(s)


def extrair_minutos_da_partida(body: dict) -> list[dict]:
    partida_info = body.get("partida", {})
    equipe1_nome = body.get("equipe1", {}).get("nome")
    equipe2_nome = body.get("equipe2", {}).get("nome")

    registros = []
    for jogador in body.get("jogadores", []):
        nome_equipe = equipe1_nome if jogador.get("equipe") == "1" else equipe2_nome
        tempos = jogador.get("tempo", [])
        for i, label in enumerate(QUARTO_LABELS):
            if i >= len(tempos):
                continue
            tempo_str = tempos[i]
            registros.append({
                "jogo_id": partida_info.get("id"),
                "data": partida_info.get("data"),
                "local": partida_info.get("local"),
                "equipe": nome_equipe,
                "jogador_id": jogador.get("id"),
                "jogador": jogador.get("nome"),
                "numero": jogador.get("numero"),
                "titular": jogador.get("titular"),
                "quarto": label,
                "tempo_mmss": tempo_str,
                "tempo_segundos": tempo_para_segundos(tempo_str),
            })
    return registros


async def coletar_links_das_partidas(page, debug: bool = False) -> list[str]:
    await page.goto(SEASON_URL_TABELA, wait_until="domcontentloaded", timeout=60000)
    await page.wait_for_timeout(3000)

    hrefs = await page.eval_on_selector_all(
        "a[href*='/partidas/']",
        "els => els.map(e => e.href)"
    )

    if debug:
        print(f"Total de hrefs brutos encontrados: {len(hrefs)}")
        print("Amostra dos primeiros 15:")
        for h in hrefs[:15]:
            print("  ", h)
        print("\nAmostra dos últimos 15:")
        for h in hrefs[-15:]:
            print("  ", h)
        # Retorna cedo em modo debug para você inspecionar sem gastar tempo
        # processando partidas
        return hrefs

    # O site tem um widget "HOJE" (jogos ao vivo/recentes de QUALQUER
    # competição/temporada) que aparece no topo de toda página e também usa
    # links /partidas/. Isso contamina a lista com jogos de 2026, LDB, etc.
    # Filtramos para manter só jogos da temporada 2022/2023 que envolvem
    # o Minas, usando o padrão textual da própria URL.
    def eh_jogo_valido(href: str) -> bool:
        href_lower = href.lower()
        eh_temporada_certa = "2022-2023" in href_lower  # cobre "nbb-2022-2023-..." e "playoffs-nbb-2022-2023-..."
        eh_minas = "minas" in href_lower
        return eh_temporada_certa and eh_minas

    vistos = set()
    links = []
    descartados = 0
    for h in hrefs:
        if not eh_jogo_valido(h):
            descartados += 1
            continue
        if h not in vistos:
            vistos.add(h)
            links.append(h)

    print(f"Encontrados {len(links)} links de partidas válidas (descartados {descartados} links irrelevantes, ex.: widget 'HOJE').")

    if not (35 <= len(links) <= 45):
        print(
            f"⚠️  Aviso: eram esperados algo em torno de 42 jogos do Minas na "
            f"temporada 2022/2023, mas foram encontrados {len(links)}. Vale "
            f"conferir manualmente (ex.: imprimir a lista `links`) antes de "
            f"rodar tudo."
        )

    return links


async def capturar_json_da_partida(context, url: str) -> dict | None:
    """Abre a página da partida e captura o JSON da API tempo_real_bybr."""
    page = await context.new_page()
    resultado = {"body": None}

    async def on_response(response):
        if "tempo_real_bybr" in response.url:
            try:
                resultado["body"] = await response.json()
            except Exception as e:
                print(f"  (erro parseando JSON de {response.url}: {e})")

    page.on("response", lambda r: asyncio.create_task(on_response(r)))

    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=60000)
        await page.wait_for_timeout(4000)  # tempo para o JS disparar a chamada da API
    except Exception as e:
        print(f"  Erro navegando para {url}: {e}")
    finally:
        await page.close()

    return resultado["body"]


async def main(limite_jogos: int | None = None):
    todos_registros = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent=USER_AGENT)
        page_lista = await context.new_page()

        links = await coletar_links_das_partidas(page_lista)
        await page_lista.close()

        if limite_jogos:
            links = links[:limite_jogos]

        for i, link in enumerate(links, 1):
            print(f"\n[{i}/{len(links)}] Processando: {link}")
            body = await capturar_json_da_partida(context, link)
            if body is None:
                print("  -> Não foi possível capturar o JSON dessa partida.")
                continue
            registros = extrair_minutos_da_partida(body)
            print(f"  -> {len(registros)} registros extraídos.")
            todos_registros.extend(registros)

            await asyncio.sleep(1)  # pequena pausa entre jogos, gentileza com o servidor

        await browser.close()

    df = pd.DataFrame(todos_registros)
    df.to_csv("minutos_por_quarto_TODOS_JOGOS.csv", index=False, encoding="utf-8-sig")
    print(f"\n\nConcluído! {len(df)} registros salvos em minutos_por_quarto_TODOS_JOGOS.csv")
    return df

In [ ]:
from playwright.async_api import async_playwright

async def debug_links():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent=USER_AGENT)
        page = await context.new_page()
        hrefs = await coletar_links_das_partidas(page, debug=True)
        await browser.close()
        return hrefs

hrefs = await debug_links()

Total de hrefs brutos encontrados: 200
Amostra dos primeiros 15:
   https://lnb.com.br/partidas/ldb-2026-campo-mourao-x-minas-21062026-0900/
   https://lnb.com.br/partidas/ldb-2026-campo-mourao-x-minas-21062026-0900/
   https://lnb.com.br/partidas/ldb-2026-uniao-corinthians-x-vasco-tijuca-21062026-0900/
   https://lnb.com.br/partidas/ldb-2026-uniao-corinthians-x-vasco-tijuca-21062026-0900/
   https://lnb.com.br/partidas/ldb-2026-basket-osasco-x-caxias-21062026-1115/
   https://lnb.com.br/partidas/ldb-2026-basket-osasco-x-caxias-21062026-1115/
   https://lnb.com.br/partidas/ldb-2026-sesi-franca-x-flamengo-21062026-1115/
   https://lnb.com.br/partidas/ldb-2026-sesi-franca-x-flamengo-21062026-1115/
   https://lnb.com.br/partidas/ldb-2026-corinthians-x-cruzeiro-21062026-1330/
   https://lnb.com.br/partidas/ldb-2026-corinthians-x-cruzeiro-21062026-1330/
   https://lnb.com.br/partidas/ldb-2026-pato-basquete-x-thalia-ph-d-esportes-21062026-1330/
   https://lnb.com.br/partidas/ldb-2026-pato-ba

In [ ]:
"""
debug_ver_mais_info.py

Investiga como o botão/link "VER MAIS INFO" de cada linha da tabela de
jogos realmente aponta para a página da partida — já que não é um
<a href="/partidas/..."> simples (confirmado no teste anterior).

Uso no Colab (depois de já ter rodado a instalação do playwright):
    resultado = await debug_ver_mais_info()
"""

from playwright.async_api import async_playwright

SEASON_URL_TABELA = (
    "https://lnb.com.br/nbb/tabela-de-jogos/"
    "?season%5B%5D=71&team%5B%5D=NTQ%3D&wherePlaying=-1&played=-1"
)

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)


async def debug_ver_mais_info():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent=USER_AGENT)
        page = await context.new_page()

        await page.goto(SEASON_URL_TABELA, wait_until="domcontentloaded", timeout=60000)
        await page.wait_for_timeout(3000)

        # 1) Procura qualquer elemento cujo texto contenha "VER MAIS INFO"
        elementos = await page.query_selector_all("text=VER MAIS INFO")
        print(f"Elementos encontrados com texto 'VER MAIS INFO': {len(elementos)}")

        for i, el in enumerate(elementos[:5]):
            tag = await el.evaluate("el => el.tagName")
            outer_html = await el.evaluate("el => el.outerHTML")
            href_proprio = await el.get_attribute("href")
            onclick = await el.get_attribute("onclick")

            # tenta achar um <a> ancestral, caso o texto esteja dentro de um link
            href_ancestral = await el.evaluate(
                "el => { const a = el.closest('a'); return a ? a.href : null }"
            )
            onclick_ancestral = await el.evaluate(
                "el => { const a = el.closest('[onclick]'); return a ? a.getAttribute('onclick') : null }"
            )

            print(f"\n--- Elemento {i} ---")
            print("Tag:", tag)
            print("href próprio:", href_proprio)
            print("onclick próprio:", onclick)
            print("href de ancestral <a>:", href_ancestral)
            print("onclick de ancestral:", onclick_ancestral)
            print("outerHTML (primeiros 400 chars):", outer_html[:400])

        # 2) Tenta clicar no primeiro e ver para onde a página navega
        if elementos:
            print("\n\n=== Testando clique no primeiro elemento ===")
            try:
                async with page.expect_navigation(timeout=8000):
                    await elementos[0].click()
                print("Navegou para:", page.url)
            except Exception as e:
                print(f"Não navegou (ou abriu nova aba/timeout): {e}")
                # Verifica se abriu uma nova aba/popup
                print("URL atual da página original:", page.url)

        await browser.close()


if __name__ == "__main__":
    import asyncio
    asyncio.run(debug_ver_mais_info())

Elementos encontrados com texto 'VER MAIS INFO': 42

--- Elemento 0 ---
Tag: TD
href próprio: None
onclick próprio: None
href de ancestral <a>: None
onclick de ancestral: None
outerHTML (primeiros 400 chars): <td class="more_info_value show-for-small-only">
                                    VER MAIS INFO
                                </td>

--- Elemento 1 ---
Tag: TD
href próprio: None
onclick próprio: None
href de ancestral <a>: None
onclick de ancestral: None
outerHTML (primeiros 400 chars): <td class="more_info_value show-for-small-only">
                                    VER MAIS INFO
                                </td>

--- Elemento 2 ---
Tag: TD
href próprio: None
onclick próprio: None
href de ancestral <a>: None
onclick de ancestral: None
outerHTML (primeiros 400 chars): <td class="more_info_value show-for-small-only">
                                    VER MAIS INFO
                                </td>

--- Elemento 3 ---
Tag: TD
href próprio: None
onclick próprio: No

In [ ]:
resultado = await debug_ver_mais_info()

Elementos encontrados com texto 'VER MAIS INFO': 42

--- Elemento 0 ---
Tag: TD
href próprio: None
onclick próprio: None
href de ancestral <a>: None
onclick de ancestral: None
outerHTML (primeiros 400 chars): <td class="more_info_value show-for-small-only">
                                    VER MAIS INFO
                                </td>

--- Elemento 1 ---
Tag: TD
href próprio: None
onclick próprio: None
href de ancestral <a>: None
onclick de ancestral: None
outerHTML (primeiros 400 chars): <td class="more_info_value show-for-small-only">
                                    VER MAIS INFO
                                </td>

--- Elemento 2 ---
Tag: TD
href próprio: None
onclick próprio: None
href de ancestral <a>: None
onclick de ancestral: None
outerHTML (primeiros 400 chars): <td class="more_info_value show-for-small-only">
                                    VER MAIS INFO
                                </td>

--- Elemento 3 ---
Tag: TD
href próprio: None
onclick próprio: No

In [ ]:
"""
debug_tr_linha.py

Investiga a linha <tr> inteira de cada jogo na tabela, procurando por
qualquer mecanismo de navegação: onclick, data-* attributes, ou links
<a> escondidos em qualquer lugar dentro da linha.

Uso no Colab:
    resultado = await debug_tr_linha()
"""

from playwright.async_api import async_playwright

SEASON_URL_TABELA = (
    "https://lnb.com.br/nbb/tabela-de-jogos/"
    "?season%5B%5D=71&team%5B%5D=NTQ%3D&wherePlaying=-1&played=-1"
)

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)


async def debug_tr_linha():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        # Usa um viewport "desktop" explícito, já que a versão mobile
        # esconde os elementos que vimos até agora
        context = await browser.new_context(
            user_agent=USER_AGENT,
            viewport={"width": 1600, "height": 1000},
        )
        page = await context.new_page()

        await page.goto(SEASON_URL_TABELA, wait_until="domcontentloaded", timeout=60000)
        await page.wait_for_timeout(3000)

        elementos = await page.query_selector_all("text=VER MAIS INFO")
        print(f"Elementos 'VER MAIS INFO' encontrados: {len(elementos)}")

        if not elementos:
            print("Nenhum elemento encontrado.")
            await browser.close()
            return

        el = elementos[0]

        # Pega a <tr> ancestral mais próxima
        info_tr = await el.evaluate("""
            el => {
                const tr = el.closest('tr');
                if (!tr) return null;
                const attrs = {};
                for (const attr of tr.attributes) {
                    attrs[attr.name] = attr.value;
                }
                // procura QUALQUER <a> dentro da linha, em qualquer profundidade
                const links = Array.from(tr.querySelectorAll('a')).map(a => a.href);
                // procura elementos com data-* que pareçam URL ou id
                const dataElements = Array.from(tr.querySelectorAll('[data-href], [data-url], [data-link], [data-id], [data-game], [data-jogo]'))
                    .map(e => ({ tag: e.tagName, attrs: Object.fromEntries(Array.from(e.attributes).map(a => [a.name, a.value])) }));
                return {
                    attrs: attrs,
                    outerHTML_preview: tr.outerHTML.slice(0, 2000),
                    links: links,
                    dataElements: dataElements,
                };
            }
        """)

        print("\n=== Atributos da <tr> ===")
        print(info_tr["attrs"])

        print("\n=== Links <a> dentro da <tr> (qualquer profundidade) ===")
        print(info_tr["links"])

        print("\n=== Elementos com data-* dentro da <tr> ===")
        for d in info_tr["dataElements"]:
            print(d)

        print("\n=== outerHTML da <tr> (primeiros 2000 chars) ===")
        print(info_tr["outerHTML_preview"])

        await browser.close()
        return info_tr


if __name__ == "__main__":
    import asyncio
    asyncio.run(debug_tr_linha())

Elementos 'VER MAIS INFO' encontrados: 42

=== Atributos da <tr> ===
{'class': 'with-hotel'}

=== Links <a> dentro da <tr> (qualquer profundidade) ===
['https://lnb.com.br/noticias/nbb-22-23-123-minas-73-x-85-flamengo/', 'https://lnb.com.br/noticias/nbb-22-23-123-minas-73-x-85-flamengo/']

=== Elementos com data-* dentro da <tr> ===

=== outerHTML da <tr> (primeiros 2000 chars) ===
<tr class="with-hotel">
                                <td class="position_value show-for-medium" data-label="JOGO" data-real-id="24475">1</td>
                                <td class="date_value show-for-medium" data-label="DATA">
                                                                            <span class="">15/10/2022</span>
                                        <span class="">18:15</span>
                                                                    </td>
                                <td class="date_value show-for-small-only" data-label="">
                                    JOG

In [ ]:
!pip install playwright nest_asyncio pandas
!playwright install chromium
!playwright install-deps chromium

Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is already the newest version (20120503-10build1

In [ ]:
"""
pipeline_final_playwright.py

Versão final e otimizada:
1. Usa Playwright para abrir a tabela de jogos UMA VEZ e extrair os
   data-real-id de cada um dos 42 jogos (requests puro leva 403 Forbidden
   por causa de proteção anti-bot do site; via Playwright funciona).
2. Usa o CONTEXTO DE REDE do navegador (context.request.get) para chamar
   diretamente a API de cada jogo — muito mais rápido que abrir 42 páginas
   completas, porque não carrega HTML/JS/imagens, só a chamada da API.
3. Parseia e junta tudo num único CSV.

Antes de rodar no Colab (uma vez por sessão):
    !pip install playwright nest_asyncio pandas
    !playwright install chromium
    !playwright install-deps chromium

Uso no Colab:
    df = await main()
"""

import nest_asyncio
nest_asyncio.apply()

from playwright.async_api import async_playwright
import asyncio
import pandas as pd

SEASON_URL_TABELA = (
    "https://lnb.com.br/nbb/tabela-de-jogos/"
    "?season%5B%5D=71&team%5B%5D=NTQ%3D&wherePlaying=-1&played=-1"
)

API_URL_TEMPLATE = "https://lnb.com.br/ws/tempo_real_bybr/json/{id}_tempo_real.json"

QUARTO_LABELS = ["Total", "1ºQ", "2ºQ", "3ºQ", "4ºQ"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)


def tempo_para_segundos(tempo_str: str) -> int:
    if not tempo_str:
        return 0
    m, s = tempo_str.split(":")
    return int(m) * 60 + int(s)


def extrair_minutos_da_partida(body: dict) -> list[dict]:
    partida_info = body.get("partida", {})
    equipe1_nome = body.get("equipe1", {}).get("nome")
    equipe2_nome = body.get("equipe2", {}).get("nome")

    registros = []
    for jogador in body.get("jogadores", []):
        nome_equipe = equipe1_nome if jogador.get("equipe") == "1" else equipe2_nome
        tempos = jogador.get("tempo", [])
        for i, label in enumerate(QUARTO_LABELS):
            if i >= len(tempos):
                continue
            tempo_str = tempos[i]
            registros.append({
                "jogo_id": partida_info.get("id"),
                "data": partida_info.get("data"),
                "local": partida_info.get("local"),
                "equipe": nome_equipe,
                "jogador_id": jogador.get("id"),
                "jogador": jogador.get("nome"),
                "numero": jogador.get("numero"),
                "titular": jogador.get("titular"),
                "quarto": label,
                "tempo_mmss": tempo_str,
                "tempo_segundos": tempo_para_segundos(tempo_str),
            })
    return registros


async def obter_ids_dos_jogos(page) -> list[str]:
    await page.goto(SEASON_URL_TABELA, wait_until="domcontentloaded", timeout=60000)
    await page.wait_for_timeout(2000)

    ids = await page.eval_on_selector_all(
        "td.position_value[data-real-id]",
        "els => els.map(e => e.getAttribute('data-real-id'))"
    )
    ids_unicos = list(dict.fromkeys(ids))

    print(f"IDs de jogos encontrados: {len(ids_unicos)}")
    if not (35 <= len(ids_unicos) <= 45):
        print(
            f"⚠️  Aviso: eram esperados ~42 jogos, mas foram encontrados "
            f"{len(ids_unicos)}. Vale conferir manualmente."
        )
    return ids_unicos


async def main(limite_jogos: int | None = None) -> pd.DataFrame:
    todos_registros = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent=USER_AGENT)
        page = await context.new_page()

        ids = await obter_ids_dos_jogos(page)
        await page.close()

        if limite_jogos:
            ids = ids[:limite_jogos]

        for i, game_id in enumerate(ids, 1):
            url = API_URL_TEMPLATE.format(id=game_id)
            print(f"[{i}/{len(ids)}] Buscando jogo {game_id}...")
            try:
                resp = await context.request.get(url, timeout=20000)
                if resp.status != 200:
                    print(f"  -> Status {resp.status}, pulando.")
                    continue
                body = await resp.json()
            except Exception as e:
                print(f"  -> Erro: {e}")
                continue

            registros = extrair_minutos_da_partida(body)
            print(f"  -> {len(registros)} registros extraídos.")
            todos_registros.extend(registros)

            await asyncio.sleep(0.3)  # gentileza com o servidor

        await browser.close()

    df = pd.DataFrame(todos_registros)
    df.to_csv("minutos_por_quarto_TODOS_JOGOS.csv", index=False, encoding="utf-8-sig")
    print(f"\n\nConcluído! {len(df)} registros salvos em minutos_por_quarto_TODOS_JOGOS.csv")
    if not df.empty:
        print(f"Jogos processados com sucesso: {df['jogo_id'].nunique()} de {len(ids)}")
    return df

In [ ]:
df = await main()

IDs de jogos encontrados: 42
[1/42] Buscando jogo 24475...
  -> 120 registros extraídos.
[2/42] Buscando jogo 24484...
  -> 115 registros extraídos.
[3/42] Buscando jogo 24487...
  -> 115 registros extraídos.
[4/42] Buscando jogo 24494...
  -> 110 registros extraídos.
[5/42] Buscando jogo 24503...
  -> 115 registros extraídos.
[6/42] Buscando jogo 24519...
  -> 115 registros extraídos.
[7/42] Buscando jogo 24524...
  -> 115 registros extraídos.
[8/42] Buscando jogo 24532...
  -> 120 registros extraídos.
[9/42] Buscando jogo 24538...
  -> 110 registros extraídos.
[10/42] Buscando jogo 24550...
  -> 115 registros extraídos.
[11/42] Buscando jogo 24556...
  -> 120 registros extraídos.
[12/42] Buscando jogo 24565...
  -> 110 registros extraídos.
[13/42] Buscando jogo 24578...
  -> 115 registros extraídos.
[14/42] Buscando jogo 24586...
  -> 115 registros extraídos.
[15/42] Buscando jogo 24595...
  -> 120 registros extraídos.
[16/42] Buscando jogo 24610...
  -> 110 registros extraídos.
[17/

In [ ]:
from google.colab import files
files.download("minutos_por_quarto_TODOS_JOGOS.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df.shape
df.head(20)

# Ver quantos jogos e atletas distintos
print(df['jogo_id'].nunique(), "jogos")
print(df['jogador'].nunique(), "jogadores distintos")

# Ver só os do Minas, por exemplo
df[df['equipe'].str.contains('Minas', case=False, na=False)].head(20)

42 jogos
214 jogadores distintos


,jogo_id,data,local,equipe,jogador_id,jogador,numero,titular,quarto,tempo_mmss,tempo_segundos
120,24484,18/10/2022 19:00,Poliesportivo H. Villaboim,123 Minas,12608,Djalo,0,False,Total,18:21,1101
121,24484,18/10/2022 19:00,Poliesportivo H. Villaboim,123 Minas,12608,Djalo,0,False,1ºQ,03:12,192
122,24484,18/10/2022 19:00,Poliesportivo H. Villaboim,123 Minas,12608,Djalo,0,False,2ºQ,07:18,438
123,24484,18/10/2022 19:00,Poliesportivo H. Villaboim,123 Minas,12608,Djalo,0,False,3ºQ,05:18,318
124,24484,18/10/2022 19:00,Poliesportivo H. Villaboim,123 Minas,12608,Djalo,0,False,4ºQ,02:33,153
125,24484,18/10/2022 19:00,Poliesportivo H. Villaboim,123 Minas,12682,J. Prado,0,False,Total,00:00,0
126,24484,18/10/2022 19:00,Poliesportivo H. Villaboim,123 Minas,12682,J. Prado,0,False,1ºQ,00:00,0
127,24484,18/10/2022 19:00,Poliesportivo H. Villaboim,123 Minas,12682,J. Prado,0,False,2ºQ,00:00,0
128,24484,18/10/2022 19:00,Poliesportivo H. Villaboim,123 Minas,12682,J. Prado,0,False,3ºQ,00:00,0
129,24484,18/10/2022 19:00,Poliesportivo H. Villaboim,123 Minas,12682,J. Prado,0,False,4ºQ,00:00,0


#Temporada 22_23

In [3]:
"""
pipeline_final_playwright.py

Versão final e otimizada:
1. Usa Playwright para abrir a tabela de jogos UMA VEZ e extrair os
   data-real-id de cada um dos 42 jogos (requests puro leva 403 Forbidden
   por causa de proteção anti-bot do site; via Playwright funciona).
2. Usa o CONTEXTO DE REDE do navegador (context.request.get) para chamar
   diretamente a API de cada jogo — muito mais rápido que abrir 42 páginas
   completas, porque não carrega HTML/JS/imagens, só a chamada da API.
3. Parseia e junta tudo num único CSV.

Antes de rodar no Colab (uma vez por sessão):
    !pip install playwright nest_asyncio pandas
    !playwright install chromium
    !playwright install-deps chromium

Uso no Colab:
    df = await main()
"""

import nest_asyncio
nest_asyncio.apply()

from playwright.async_api import async_playwright
import asyncio
import pandas as pd

SEASON_URL_TABELA = (
    "https://lnb.com.br/nbb/tabela-de-jogos/"
    "?season%5B%5D=71&team%5B%5D=NTQ%3D&wherePlaying=-1&played=-1"
)

API_URL_TEMPLATE = "https://lnb.com.br/ws/tempo_real_bybr/json/{id}_tempo_real.json"

QUARTO_LABELS = ["Total", "1ºQ", "2ºQ", "3ºQ", "4ºQ"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)


def tempo_para_segundos(tempo_str: str) -> int:
    if not tempo_str:
        return 0
    m, s = tempo_str.split(":")
    return int(m) * 60 + int(s)


def extrair_minutos_da_partida(body: dict) -> list[dict]:
    partida_info = body.get("partida", {})
    equipe1 = body.get("equipe1", {})
    equipe2 = body.get("equipe2", {})
    equipe1_id = equipe1.get("id")
    equipe1_nome = equipe1.get("nome")
    equipe2_nome = equipe2.get("nome")

    registros = []
    for jogador in body.get("jogadores", []):
        # O campo "equipe" do jogador contém o ID do time (igual a
        # equipe1["id"] ou equipe2["id"]), NÃO um literal "1"/"2" fixo.
        # Por isso comparamos com o ID real do equipe1, em vez de "1".
        nome_equipe = equipe1_nome if jogador.get("equipe") == equipe1_id else equipe2_nome
        tempos = jogador.get("tempo", [])
        for i, label in enumerate(QUARTO_LABELS):
            if i >= len(tempos):
                continue
            tempo_str = tempos[i]
            registros.append({
                "jogo_id": partida_info.get("id"),
                "data": partida_info.get("data"),
                "local": partida_info.get("local"),
                "equipe": nome_equipe,
                "jogador_id": jogador.get("id"),
                "jogador": jogador.get("nome"),
                "numero": jogador.get("numero"),
                "titular": jogador.get("titular"),
                "quarto": label,
                "tempo_mmss": tempo_str,
                "tempo_segundos": tempo_para_segundos(tempo_str),
            })
    return registros


async def obter_ids_dos_jogos(page) -> list[str]:
    await page.goto(SEASON_URL_TABELA, wait_until="domcontentloaded", timeout=60000)
    await page.wait_for_timeout(2000)

    ids = await page.eval_on_selector_all(
        "td.position_value[data-real-id]",
        "els => els.map(e => e.getAttribute('data-real-id'))"
    )
    ids_unicos = list(dict.fromkeys(ids))

    print(f"IDs de jogos encontrados: {len(ids_unicos)}")
    if not (35 <= len(ids_unicos) <= 45):
        print(
            f"⚠️  Aviso: eram esperados ~42 jogos, mas foram encontrados "
            f"{len(ids_unicos)}. Vale conferir manualmente."
        )
    return ids_unicos


async def main(limite_jogos: int | None = None) -> pd.DataFrame:
    todos_registros = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent=USER_AGENT)
        page = await context.new_page()

        ids = await obter_ids_dos_jogos(page)
        await page.close()

        if limite_jogos:
            ids = ids[:limite_jogos]

        for i, game_id in enumerate(ids, 1):
            url = API_URL_TEMPLATE.format(id=game_id)
            print(f"[{i}/{len(ids)}] Buscando jogo {game_id}...")
            try:
                resp = await context.request.get(url, timeout=20000)
                if resp.status != 200:
                    print(f"  -> Status {resp.status}, pulando.")
                    continue
                body = await resp.json()
            except Exception as e:
                print(f"  -> Erro: {e}")
                continue

            registros = extrair_minutos_da_partida(body)
            print(f"  -> {len(registros)} registros extraídos.")
            todos_registros.extend(registros)

            await asyncio.sleep(0.3)  # gentileza com o servidor

        await browser.close()

    df = pd.DataFrame(todos_registros)
    df.to_csv("minutos_por_quarto_TODOS_JOGOS.csv", index=False, encoding="utf-8-sig")
    print(f"\n\nConcluído! {len(df)} registros salvos em minutos_por_quarto_22_23.csv")
    if not df.empty:
        print(f"Jogos processados com sucesso: {df['jogo_id'].nunique()} de {len(ids)}")
    return df

In [4]:
df = await main()

IDs de jogos encontrados: 42
[1/42] Buscando jogo 24475...
  -> 120 registros extraídos.
[2/42] Buscando jogo 24484...
  -> 115 registros extraídos.
[3/42] Buscando jogo 24487...
  -> 115 registros extraídos.
[4/42] Buscando jogo 24494...
  -> 110 registros extraídos.
[5/42] Buscando jogo 24503...
  -> 115 registros extraídos.
[6/42] Buscando jogo 24519...
  -> 115 registros extraídos.
[7/42] Buscando jogo 24524...
  -> 115 registros extraídos.
[8/42] Buscando jogo 24532...
  -> 120 registros extraídos.
[9/42] Buscando jogo 24538...
  -> 110 registros extraídos.
[10/42] Buscando jogo 24550...
  -> 115 registros extraídos.
[11/42] Buscando jogo 24556...
  -> 120 registros extraídos.
[12/42] Buscando jogo 24565...
  -> 110 registros extraídos.
[13/42] Buscando jogo 24578...
  -> 115 registros extraídos.
[14/42] Buscando jogo 24586...
  -> 115 registros extraídos.
[15/42] Buscando jogo 24595...
  -> 120 registros extraídos.
[16/42] Buscando jogo 24610...
  -> 110 registros extraídos.
[17/

#Temporada 23_24

In [11]:
"""
pipeline_final_playwright.py

Versão final e otimizada:
1. Usa Playwright para abrir a tabela de jogos UMA VEZ e extrair os
   data-real-id de cada um dos 42 jogos (requests puro leva 403 Forbidden
   por causa de proteção anti-bot do site; via Playwright funciona).
2. Usa o CONTEXTO DE REDE do navegador (context.request.get) para chamar
   diretamente a API de cada jogo — muito mais rápido que abrir 42 páginas
   completas, porque não carrega HTML/JS/imagens, só a chamada da API.
3. Parseia e junta tudo num único CSV.

Antes de rodar no Colab (uma vez por sessão):
    !pip install playwright nest_asyncio pandas
    !playwright install chromium
    !playwright install-deps chromium

Uso no Colab:
    df = await main()
"""

import nest_asyncio
nest_asyncio.apply()

from playwright.async_api import async_playwright
import asyncio
import pandas as pd

SEASON_URL_TABELA = (
    "https://lnb.com.br/nbb/tabela-de-jogos/"
    "?season%5B%5D=80&team%5B%5D=NTQ%3D"
)

API_URL_TEMPLATE = "https://lnb.com.br/ws/tempo_real_bybr/json/{id}_tempo_real.json"

QUARTO_LABELS = ["Total", "1ºQ", "2ºQ", "3ºQ", "4ºQ"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)


def tempo_para_segundos(tempo_str: str) -> int:
    if not tempo_str:
        return 0
    m, s = tempo_str.split(":")
    return int(m) * 60 + int(s)


def extrair_minutos_da_partida(body: dict) -> list[dict]:
    partida_info = body.get("partida", {})
    equipe1 = body.get("equipe1", {})
    equipe2 = body.get("equipe2", {})
    equipe1_id = equipe1.get("id")
    equipe1_nome = equipe1.get("nome")
    equipe2_nome = equipe2.get("nome")

    registros = []
    for jogador in body.get("jogadores", []):
        # O campo "equipe" do jogador contém o ID do time (igual a
        # equipe1["id"] ou equipe2["id"]), NÃO um literal "1"/"2" fixo.
        # Por isso comparamos com o ID real do equipe1, em vez de "1".
        nome_equipe = equipe1_nome if jogador.get("equipe") == equipe1_id else equipe2_nome
        tempos = jogador.get("tempo", [])
        for i, label in enumerate(QUARTO_LABELS):
            if i >= len(tempos):
                continue
            tempo_str = tempos[i]
            registros.append({
                "jogo_id": partida_info.get("id"),
                "data": partida_info.get("data"),
                "local": partida_info.get("local"),
                "equipe": nome_equipe,
                "jogador_id": jogador.get("id"),
                "jogador": jogador.get("nome"),
                "numero": jogador.get("numero"),
                "titular": jogador.get("titular"),
                "quarto": label,
                "tempo_mmss": tempo_str,
                "tempo_segundos": tempo_para_segundos(tempo_str),
            })
    return registros


async def obter_ids_dos_jogos(page) -> list[str]:
    await page.goto(SEASON_URL_TABELA, wait_until="domcontentloaded", timeout=60000)
    await page.wait_for_timeout(2000)

    ids = await page.eval_on_selector_all(
        "td.position_value[data-real-id]",
        "els => els.map(e => e.getAttribute('data-real-id'))"
    )
    ids_unicos = list(dict.fromkeys(ids))

    print(f"IDs de jogos encontrados: {len(ids_unicos)}")
    if not (48 <= len(ids_unicos) <= 48):
        print(
            f"⚠️  Aviso: eram esperados ~48 jogos, mas foram encontrados "
            f"{len(ids_unicos)}. Vale conferir manualmente."
        )
    return ids_unicos


async def main(limite_jogos: int | None = None) -> pd.DataFrame:
    todos_registros = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent=USER_AGENT)
        page = await context.new_page()

        ids = await obter_ids_dos_jogos(page)
        await page.close()

        if limite_jogos:
            ids = ids[:limite_jogos]

        for i, game_id in enumerate(ids, 1):
            url = API_URL_TEMPLATE.format(id=game_id)
            print(f"[{i}/{len(ids)}] Buscando jogo {game_id}...")
            try:
                resp = await context.request.get(url, timeout=20000)
                if resp.status != 200:
                    print(f"  -> Status {resp.status}, pulando.")
                    continue
                body = await resp.json()
            except Exception as e:
                print(f"  -> Erro: {e}")
                continue

            registros = extrair_minutos_da_partida(body)
            print(f"  -> {len(registros)} registros extraídos.")
            todos_registros.extend(registros)

            await asyncio.sleep(0.3)  # gentileza com o servidor

        await browser.close()

    df = pd.DataFrame(todos_registros)
    df.to_csv("minutos_por_quarto_TODOS_JOGOS.csv", index=False, encoding="utf-8-sig")
    print(f"\n\nConcluído! {len(df)} registros salvos em minutos_por_quarto_23_24.csv")
    if not df.empty:
        print(f"Jogos processados com sucesso: {df['jogo_id'].nunique()} de {len(ids)}")
    return df

In [12]:
df = await main()

IDs de jogos encontrados: 48
⚠️  Aviso: eram esperados ~42 jogos, mas foram encontrados 48. Vale conferir manualmente.
[1/48] Buscando jogo 25066...
  -> 115 registros extraídos.
[2/48] Buscando jogo 25071...
  -> 95 registros extraídos.
[3/48] Buscando jogo 25076...
  -> 115 registros extraídos.
[4/48] Buscando jogo 25103...
  -> 115 registros extraídos.
[5/48] Buscando jogo 25111...
  -> 115 registros extraídos.
[6/48] Buscando jogo 25126...
  -> 110 registros extraídos.
[7/48] Buscando jogo 25131...
  -> 110 registros extraídos.
[8/48] Buscando jogo 25144...
  -> 115 registros extraídos.
[9/48] Buscando jogo 25151...
  -> 115 registros extraídos.
[10/48] Buscando jogo 25158...
  -> 115 registros extraídos.
[11/48] Buscando jogo 25165...
  -> 115 registros extraídos.
[12/48] Buscando jogo 25188...
  -> 110 registros extraídos.
[13/48] Buscando jogo 25204...
  -> 115 registros extraídos.
[14/48] Buscando jogo 25206...
  -> 115 registros extraídos.
[15/48] Buscando jogo 25214...
  -> 1

#Temporada 24_25

In [9]:
"""
pipeline_final_playwright.py

Versão final e otimizada:
1. Usa Playwright para abrir a tabela de jogos UMA VEZ e extrair os
   data-real-id de cada um dos 42 jogos (requests puro leva 403 Forbidden
   por causa de proteção anti-bot do site; via Playwright funciona).
2. Usa o CONTEXTO DE REDE do navegador (context.request.get) para chamar
   diretamente a API de cada jogo — muito mais rápido que abrir 42 páginas
   completas, porque não carrega HTML/JS/imagens, só a chamada da API.
3. Parseia e junta tudo num único CSV.

Antes de rodar no Colab (uma vez por sessão):
    !pip install playwright nest_asyncio pandas
    !playwright install chromium
    !playwright install-deps chromium

Uso no Colab:
    df = await main()
"""

import nest_asyncio
nest_asyncio.apply()

from playwright.async_api import async_playwright
import asyncio
import pandas as pd

SEASON_URL_TABELA = (
    "https://lnb.com.br/nbb/tabela-de-jogos/"
    "?season%5B%5D=88&team%5B%5D=NTQ%3D"
)

API_URL_TEMPLATE = "https://lnb.com.br/ws/tempo_real_bybr/json/{id}_tempo_real.json"

QUARTO_LABELS = ["Total", "1ºQ", "2ºQ", "3ºQ", "4ºQ"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)


def tempo_para_segundos(tempo_str: str) -> int:
    if not tempo_str:
        return 0
    m, s = tempo_str.split(":")
    return int(m) * 60 + int(s)


def extrair_minutos_da_partida(body: dict) -> list[dict]:
    partida_info = body.get("partida", {})
    equipe1 = body.get("equipe1", {})
    equipe2 = body.get("equipe2", {})
    equipe1_id = equipe1.get("id")
    equipe1_nome = equipe1.get("nome")
    equipe2_nome = equipe2.get("nome")

    registros = []
    for jogador in body.get("jogadores", []):
        # O campo "equipe" do jogador contém o ID do time (igual a
        # equipe1["id"] ou equipe2["id"]), NÃO um literal "1"/"2" fixo.
        # Por isso comparamos com o ID real do equipe1, em vez de "1".
        nome_equipe = equipe1_nome if jogador.get("equipe") == equipe1_id else equipe2_nome
        tempos = jogador.get("tempo", [])
        for i, label in enumerate(QUARTO_LABELS):
            if i >= len(tempos):
                continue
            tempo_str = tempos[i]
            registros.append({
                "jogo_id": partida_info.get("id"),
                "data": partida_info.get("data"),
                "local": partida_info.get("local"),
                "equipe": nome_equipe,
                "jogador_id": jogador.get("id"),
                "jogador": jogador.get("nome"),
                "numero": jogador.get("numero"),
                "titular": jogador.get("titular"),
                "quarto": label,
                "tempo_mmss": tempo_str,
                "tempo_segundos": tempo_para_segundos(tempo_str),
            })
    return registros


async def obter_ids_dos_jogos(page) -> list[str]:
    await page.goto(SEASON_URL_TABELA, wait_until="domcontentloaded", timeout=60000)
    await page.wait_for_timeout(2000)

    ids = await page.eval_on_selector_all(
        "td.position_value[data-real-id]",
        "els => els.map(e => e.getAttribute('data-real-id'))"
    )
    ids_unicos = list(dict.fromkeys(ids))

    print(f"IDs de jogos encontrados: {len(ids_unicos)}")
    if not (47 <= len(ids_unicos) <= 47):
        print(
            f"⚠️  Aviso: eram esperados ~47 jogos, mas foram encontrados "
            f"{len(ids_unicos)}. Vale conferir manualmente."
        )
    return ids_unicos


async def main(limite_jogos: int | None = None) -> pd.DataFrame:
    todos_registros = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent=USER_AGENT)
        page = await context.new_page()

        ids = await obter_ids_dos_jogos(page)
        await page.close()

        if limite_jogos:
            ids = ids[:limite_jogos]

        for i, game_id in enumerate(ids, 1):
            url = API_URL_TEMPLATE.format(id=game_id)
            print(f"[{i}/{len(ids)}] Buscando jogo {game_id}...")
            try:
                resp = await context.request.get(url, timeout=20000)
                if resp.status != 200:
                    print(f"  -> Status {resp.status}, pulando.")
                    continue
                body = await resp.json()
            except Exception as e:
                print(f"  -> Erro: {e}")
                continue

            registros = extrair_minutos_da_partida(body)
            print(f"  -> {len(registros)} registros extraídos.")
            todos_registros.extend(registros)

            await asyncio.sleep(0.3)  # gentileza com o servidor

        await browser.close()

    df = pd.DataFrame(todos_registros)
    df.to_csv("minutos_por_quarto_TODOS_JOGOS.csv", index=False, encoding="utf-8-sig")
    print(f"\n\nConcluído! {len(df)} registros salvos em minutos_por_quarto_24_25.csv")
    if not df.empty:
        print(f"Jogos processados com sucesso: {df['jogo_id'].nunique()} de {len(ids)}")
    return df

In [10]:
df = await main()

IDs de jogos encontrados: 47
⚠️  Aviso: eram esperados ~42 jogos, mas foram encontrados 47. Vale conferir manualmente.
[1/47] Buscando jogo 25741...
  -> 110 registros extraídos.
[2/47] Buscando jogo 25751...
  -> 110 registros extraídos.
[3/47] Buscando jogo 25753...
  -> 115 registros extraídos.
[4/47] Buscando jogo 25761...
  -> 110 registros extraídos.
[5/47] Buscando jogo 25768...
  -> 115 registros extraídos.
[6/47] Buscando jogo 25780...
  -> 110 registros extraídos.
[7/47] Buscando jogo 25784...
  -> 115 registros extraídos.
[8/47] Buscando jogo 25802...
  -> 120 registros extraídos.
[9/47] Buscando jogo 25811...
  -> 115 registros extraídos.
[10/47] Buscando jogo 25821...
  -> 115 registros extraídos.
[11/47] Buscando jogo 25827...
  -> 115 registros extraídos.
[12/47] Buscando jogo 25840...
  -> 115 registros extraídos.
[13/47] Buscando jogo 25844...
  -> 115 registros extraídos.
[14/47] Buscando jogo 25848...
  -> 110 registros extraídos.
[15/47] Buscando jogo 25859...
  -> 

#Temporada 25_26

In [6]:
"""
pipeline_final_playwright.py

Versão final e otimizada:
1. Usa Playwright para abrir a tabela de jogos UMA VEZ e extrair os
   data-real-id de cada um dos 42 jogos (requests puro leva 403 Forbidden
   por causa de proteção anti-bot do site; via Playwright funciona).
2. Usa o CONTEXTO DE REDE do navegador (context.request.get) para chamar
   diretamente a API de cada jogo — muito mais rápido que abrir 42 páginas
   completas, porque não carrega HTML/JS/imagens, só a chamada da API.
3. Parseia e junta tudo num único CSV.

Antes de rodar no Colab (uma vez por sessão):
    !pip install playwright nest_asyncio pandas
    !playwright install chromium
    !playwright install-deps chromium

Uso no Colab:
    df = await main()
"""

import nest_asyncio
nest_asyncio.apply()

from playwright.async_api import async_playwright
import asyncio
import pandas as pd

SEASON_URL_TABELA = (
    "https://lnb.com.br/nbb/tabela-de-jogos/"
    "?season%5B%5D=97&team%5B%5D=NTQ%3D&wherePlaying=-1&played=-1"
)

API_URL_TEMPLATE = "https://lnb.com.br/ws/tempo_real_bybr/json/{id}_tempo_real.json"

QUARTO_LABELS = ["Total", "1ºQ", "2ºQ", "3ºQ", "4ºQ"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)


def tempo_para_segundos(tempo_str: str) -> int:
    if not tempo_str:
        return 0
    m, s = tempo_str.split(":")
    return int(m) * 60 + int(s)


def extrair_minutos_da_partida(body: dict) -> list[dict]:
    partida_info = body.get("partida", {})
    equipe1 = body.get("equipe1", {})
    equipe2 = body.get("equipe2", {})
    equipe1_id = equipe1.get("id")
    equipe1_nome = equipe1.get("nome")
    equipe2_nome = equipe2.get("nome")

    registros = []
    for jogador in body.get("jogadores", []):
        # O campo "equipe" do jogador contém o ID do time (igual a
        # equipe1["id"] ou equipe2["id"]), NÃO um literal "1"/"2" fixo.
        # Por isso comparamos com o ID real do equipe1, em vez de "1".
        nome_equipe = equipe1_nome if jogador.get("equipe") == equipe1_id else equipe2_nome
        tempos = jogador.get("tempo", [])
        for i, label in enumerate(QUARTO_LABELS):
            if i >= len(tempos):
                continue
            tempo_str = tempos[i]
            registros.append({
                "jogo_id": partida_info.get("id"),
                "data": partida_info.get("data"),
                "local": partida_info.get("local"),
                "equipe": nome_equipe,
                "jogador_id": jogador.get("id"),
                "jogador": jogador.get("nome"),
                "numero": jogador.get("numero"),
                "titular": jogador.get("titular"),
                "quarto": label,
                "tempo_mmss": tempo_str,
                "tempo_segundos": tempo_para_segundos(tempo_str),
            })
    return registros


async def obter_ids_dos_jogos(page) -> list[str]:
    await page.goto(SEASON_URL_TABELA, wait_until="domcontentloaded", timeout=60000)
    await page.wait_for_timeout(2000)

    ids = await page.eval_on_selector_all(
        "td.position_value[data-real-id]",
        "els => els.map(e => e.getAttribute('data-real-id'))"
    )
    ids_unicos = list(dict.fromkeys(ids))

    print(f"IDs de jogos encontrados: {len(ids_unicos)}")
    if not (46 <= len(ids_unicos) <= 46):
        print(
            f"⚠️  Aviso: eram esperados ~46 jogos, mas foram encontrados "
            f"{len(ids_unicos)}. Vale conferir manualmente."
        )
    return ids_unicos


async def main(limite_jogos: int | None = None) -> pd.DataFrame:
    todos_registros = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent=USER_AGENT)
        page = await context.new_page()

        ids = await obter_ids_dos_jogos(page)
        await page.close()

        if limite_jogos:
            ids = ids[:limite_jogos]

        for i, game_id in enumerate(ids, 1):
            url = API_URL_TEMPLATE.format(id=game_id)
            print(f"[{i}/{len(ids)}] Buscando jogo {game_id}...")
            try:
                resp = await context.request.get(url, timeout=20000)
                if resp.status != 200:
                    print(f"  -> Status {resp.status}, pulando.")
                    continue
                body = await resp.json()
            except Exception as e:
                print(f"  -> Erro: {e}")
                continue

            registros = extrair_minutos_da_partida(body)
            print(f"  -> {len(registros)} registros extraídos.")
            todos_registros.extend(registros)

            await asyncio.sleep(0.3)  # gentileza com o servidor

        await browser.close()

    df = pd.DataFrame(todos_registros)
    df.to_csv("minutos_por_quarto_TODOS_JOGOS.csv", index=False, encoding="utf-8-sig")
    print(f"\n\nConcluído! {len(df)} registros salvos em minutos_por_quarto_25_26.csv")
    if not df.empty:
        print(f"Jogos processados com sucesso: {df['jogo_id'].nunique()} de {len(ids)}")
    return df

In [8]:
df = await main()

IDs de jogos encontrados: 46
⚠️  Aviso: eram esperados ~42 jogos, mas foram encontrados 46. Vale conferir manualmente.
[1/46] Buscando jogo 26369...
  -> 115 registros extraídos.
[2/46] Buscando jogo 26373...
  -> 110 registros extraídos.
[3/46] Buscando jogo 26382...
  -> 115 registros extraídos.
[4/46] Buscando jogo 26392...
  -> 110 registros extraídos.
[5/46] Buscando jogo 26409...
  -> 120 registros extraídos.
[6/46] Buscando jogo 26416...
  -> 110 registros extraídos.
[7/46] Buscando jogo 26428...
  -> 110 registros extraídos.
[8/46] Buscando jogo 26435...
  -> 115 registros extraídos.
[9/46] Buscando jogo 26447...
  -> 120 registros extraídos.
[10/46] Buscando jogo 26455...
  -> 115 registros extraídos.
[11/46] Buscando jogo 26465...
  -> 115 registros extraídos.
[12/46] Buscando jogo 26470...
  -> 110 registros extraídos.
[13/46] Buscando jogo 26482...
  -> 115 registros extraídos.
[14/46] Buscando jogo 26490...
  -> 115 registros extraídos.
[15/46] Buscando jogo 26518...
  -> 